In [ ]:
# ==========================================================
# CELL 1 — Setup (brain, tools, tracer — loaded once, reused for every dialogue)
# ==========================================================
from brain import Brain
from agent import Agent
from tracer import Tracer
from tools import ALL_TOOLS
from memory.memory_manager import MemoryManager
import config

brain = Brain(TOKEN)

def local_generate(prompt):
    return brain.think([{"role": "user", "content": prompt}])["text"]

In [ ]:
# ==========================================================
# CELL 2 — Load the full dataset + balanced question sample
# ==========================================================
import pandas as pd

questions_df = pd.read_csv("beam_100k_questions.csv")
histories_df = pd.read_csv("beam_100k_histories.csv")

# 1 question per type per dialogue = 10 questions x 20 dialogues = 200 total,
# full category coverage, half the volume of the full 400.
questions_df = questions_df.groupby(["conversation_id", "question_type"]).head(1)

conv_ids = histories_df["conversation_id"].unique()
print("Dialogues:", len(conv_ids), "| Questions:", len(questions_df))

In [ ]:
# ==========================================================
# CELL 3 — Timing pilot: check ONE dialogue's ingestion cost before running all 20
# ==========================================================
import time

pilot_history = histories_df.iloc[0]["conversation"]
print("Pilot conversation length:", len(pilot_history), "chars")

start = time.time()
from memory.fact_extractor import extract_facts_long
pilot_facts = extract_facts_long(pilot_history, local_generate, chunk_size=9000)
elapsed = time.time() - start

print(f"Extracted {len(pilot_facts)} facts in {elapsed:.1f}s "
      f"(~{elapsed/60:.1f} min). Full run estimate: ~{elapsed*20/60:.1f} min for all 20 dialogues.")

In [ ]:
# ==========================================================
# CELL 4 — Judge (no LLM call — abstention markers + rubric keyword match)
# ==========================================================
import ast

def judge_abstention(agent_answer):
    lower = agent_answer.lower()
    return any(marker in lower for marker in config.ABSTENTION_MARKERS)

def judge_with_rubric(agent_answer, rubric_raw):
    try:
        rubric = ast.literal_eval(rubric_raw) if isinstance(rubric_raw, str) else rubric_raw
    except Exception:
        rubric = [str(rubric_raw)]

    if not isinstance(rubric, list) or len(rubric) == 0:
        return None

    answer_lower = agent_answer.lower()
    hits = sum(1 for point in rubric if str(point).lower()[:20] in answer_lower)
    return hits >= max(1, len(rubric) // 2)

def judge(q_type, agent_answer, rubric_raw):
    if q_type == "abstention":
        return judge_abstention(agent_answer)
    return judge_with_rubric(agent_answer, rubric_raw)

In [ ]:
# ==========================================================
# CELL 5 — Main eval loop: isolated memory per dialogue + checkpointing
# ==========================================================
import os
import shutil

results_file = "beam_eval_results.csv"
already_done = set()
if os.path.exists(results_file):
    already_done = set(pd.read_csv(results_file)["question_id"])

BASE_DATA_FOLDER = config.DATA_FOLDER

for conv_id in conv_ids:
    conv_questions = questions_df[questions_df["conversation_id"] == conv_id]
    if conv_questions["question_id"].isin(already_done).all():
        print(f"Dialogue {conv_id}: already fully done, skipping.")
        continue

    print(f"\n=== Dialogue {conv_id} ===")

    conv_folder = f"{BASE_DATA_FOLDER}/conv_{conv_id}"
    if os.path.exists(conv_folder):
        shutil.rmtree(conv_folder)
    config.DATA_FOLDER = conv_folder

    memory = MemoryManager()
    tracer = Tracer()
    agent = Agent(brain, memory, ALL_TOOLS, tracer)

    history_row = histories_df[histories_df["conversation_id"] == conv_id].iloc[0]

    ingest_start = time.time()
    memory.add_to_historical(history_row["conversation"], generate_fn=local_generate, auto_extract=True)
    print(f"  Ingested in {time.time() - ingest_start:.1f}s")

    for _, row in conv_questions.iterrows():
        if row["question_id"] in already_done:
            continue

        start = time.time()
        agent_answer = agent.ask(row["question"])
        seconds = time.time() - start

        correct = judge(row["question_type"], agent_answer, row["rubric"])

        result_row = pd.DataFrame([{
            "question_id": row["question_id"],
            "conversation_id": conv_id,
            "q_type": row["question_type"],
            "question": row["question"],
            "gold_answer": row["gold_answer"],
            "agent_answer": agent_answer,
            "correct": correct,
            "seconds": round(seconds, 2),
        }])
        result_row.to_csv(results_file, index=False, mode="a", header=not os.path.exists(results_file))

        print(f"  [{row['question_type']}] correct={correct} ({seconds:.1f}s)")

config.DATA_FOLDER = BASE_DATA_FOLDER
print("\nDone.")

In [ ]:
# ==========================================================
# CELL 6 — Score report
# ==========================================================
results_df = pd.read_csv(results_file)
scored = results_df[results_df["correct"].notna()]

overall = scored["correct"].mean() * 100
print(f"Overall: {overall:.1f}%  ({scored['correct'].sum()}/{len(scored)})")
print()
print(results_df.groupby("q_type")["correct"].agg(["mean", "count"]))